# AI Startup Idea Validator 

This notebook builds up a startup-idea checker 


In [8]:
# installing the Google Gen AI SDK (this is the library that talks to Gemini)
# %pip is used instead of !pip so it installs into the SAME python this notebook is using
%pip install google-genai pandas ipywidgets --quiet
print("done installing")


Note: you may need to restart the kernel to use updated packages.
done installing


In [9]:
# basic python stuff we'll need
import os
import json
import re
import getpass
from datetime import datetime


In [10]:
# this is the actual Gemini library
from google import genai
from google.genai import types

print("genai imported ok")


genai imported ok


In [11]:
# for showing nicer output and a little UI later
import pandas as pd
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets


## Step 0: connect to Gemini

If you've set the key as an environment variable (`GEMINI_API_KEY`) it'll just use that.
Otherwise it asks you to paste it in (hidden, like a password field).


In [12]:
my_api_key = os.environ.get("GEMINI_API_KEY")

if not my_api_key:
    my_api_key = getpass.getpass("Paste your Gemini API key here: ")

client = genai.Client(api_key=my_api_key)

# Gemini has a few model sizes. Flash is fast and cheap (and works on the free tier).
# Pro thinks harder but currently isn't free. We'll use Flash.
MODEL = "gemini-2.5-flash"

print("client ready, using model:", MODEL)


Paste your Gemini API key here:  ········


client ready, using model: gemini-2.5-flash


## Step 1: does it even work?

Before building anything fancy, just send one message and see what comes back.


In [13]:
test_response = client.models.generate_content(
    model=MODEL,
    contents="Say hello in one sentence and confirm you're working.",
)

print(test_response.text)


Hello there, I am working and ready to assist you.


## Step 2: try a naive startup-evaluation prompt

Let's just ask it directly with no structure and see what we get.


In [16]:
my_idea = "An AI app that listens to customer support calls and auto-writes the follow-up email."

basic_prompt = f"Is this a good startup idea? {my_idea}"

response = client.models.generate_content(
    model=MODEL,
    contents=basic_prompt,
)

print(response.text)


This is a **very promising idea with strong market potential, but also significant technical, ethical, and competitive challenges.**

Let's break down why:

## The Good (Strong Potential)

1.  **Clear Pain Point:** Customer support agents spend a considerable amount of time post-call summarizing, documenting, and drafting follow-up emails. This is repetitive, time-consuming, and prone to human error.
2.  **Efficiency Gains:** Automating this task can save agents significant time, allowing them to handle more calls, reduce wrap-up time, and focus on higher-value customer interactions.
3.  **Consistency & Quality:** AI can ensure follow-up emails are consistently formatted, adhere to brand guidelines, include all necessary information, and maintain a professional tone. This improves customer experience.
4.  **Cost Reduction:** For large call centers, even marginal time savings per call can translate into substantial operational cost reductions.
5.  **Agent Satisfaction:** Reducing tediou

This kind of works, but the answer is just a paragraph of opinion. There's no score, no
breakdown, and if you ran it twice you'd get differently-shaped answers each time. That makes it
useless for comparing ideas side by side. Let's fix that next.


## Step 3: add a scoring rubric

Telling the model exactly what to score, on what scale, makes answers comparable.


In [17]:
rubric_prompt = f"""Evaluate this startup idea and score it 1-10 on each of these:
1. Problem-solution fit
2. Market opportunity
3. Competitive landscape
4. Business model
5. Technical feasibility (for a generative AI product)
6. Moat / how hard it is to copy

Idea: {my_idea}
"""

response = client.models.generate_content(
    model=MODEL,
    contents=rubric_prompt,
)

print(response.text)


This is a strong idea with clear potential. Let's break it down:

---

### Startup Idea: AI app that listens to customer support calls and auto-writes the follow-up email.

---

### Evaluation:

1.  **Problem-solution fit: 9/10**
    *   **Problem:** Customer support agents spend significant time post-call summarizing, drafting, and editing follow-up emails. This leads to inefficiency, inconsistent email quality, potential for human error (missing key details), and delayed customer communication.
    *   **Solution:** Directly addresses the time sink and quality issues. By automating the draft, agents can review, make minor edits, and send much faster. This frees up agent time for more calls, ensures consistency, and improves overall customer experience.
    *   **Fit:** Excellent. The solution directly solves a quantifiable pain point for a large demographic (customer service agents and their managers).

2.  **Market opportunity: 9/10**
    *   **Target Market:** Any business with a c

Better — now there are actual numbers. But the output is still free-form text, so our
code can't easily pull "6/10" out of a paragraph reliably. Next step: ask for JSON instead of prose.


## Step 4: ask for JSON instead of plain text

If we tell it exactly what JSON shape we want, we can `json.loads()` the answer straight into
a python dictionary.


In [19]:
json_prompt = f"""Evaluate this startup idea. Score 1-10 on each category below.
Respond with ONLY valid JSON, no extra text, no markdown fences. Use this shape:

{{
  "problem_solution_fit": <int 1-10>,
  "market_opportunity": <int 1-10>,
  "competitive_landscape": <int 1-10>,
  "business_model": <int 1-10>,
  "technical_feasibility": <int 1-10>,
  "moat": <int 1-10>,
  "overall_comment": "<one sentence>"
}}

Idea: {my_idea}
"""

response = client.models.generate_content(
    model=MODEL,
    contents=json_prompt,
)

print(response.text)

# let's see if it actually parses
try:
    parsed = json.loads(response.text)
    print("\nParsed fine:", parsed)
except json.JSONDecodeError as e:
    print("\nDidn't parse cleanly:", e)


{
  "problem_solution_fit": 9,
  "market_opportunity": 9,
  "competitive_landscape": 6,
  "business_model": 9,
  "technical_feasibility": 8,
  "moat": 6,
  "overall_comment": "This idea offers a compelling solution to a significant operational pain point in a large market, but building a defensible moat against well-funded incumbents will be critical."
}

Parsed fine: {'problem_solution_fit': 9, 'market_opportunity': 9, 'competitive_landscape': 6, 'business_model': 9, 'technical_feasibility': 8, 'moat': 6, 'overall_comment': 'This idea offers a compelling solution to a significant operational pain point in a large market, but building a defensible moat against well-funded incumbents will be critical.'}


Models sometimes wrap JSON in ` ```json ` fences even when you ask them not to, so the
parsing above can fail occasionally. We'll write a small helper that strips fences and grabs the
`{...}` block, so it parses reliably whether or not the model adds fences.


In [20]:
def extract_json(text):
    """Pulls a JSON object out of model output, even if it's wrapped in code fences or has stray text around it."""
    cleaned = text.strip()
    # remove ```json or ``` fences if present
    cleaned = re.sub(r"^```(json)?", "", cleaned)
    cleaned = re.sub(r"```$", "", cleaned)
    cleaned = cleaned.strip()
    # find the { ... } block in case there's extra text before/after it
    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if not match:
        raise ValueError("couldn't find a JSON object in the response")
    return json.loads(match.group(0))


# quick test with a fake messy response to make sure it works
fake_text = """Sure, here you go:
```json
{"score": 7}
```
"""
print(extract_json(fake_text))


{'score': 7}


## Step 5: give it live web search

For market size and competitors, the model's training data can be stale. Gemini has a built-in
Google Search tool we can turn on so it looks things up before answering.


In [21]:
search_tool = types.Tool(google_search=types.GoogleSearch())

search_prompt = f"Who are the current competitors for this idea, and roughly how big is the market? Idea: {my_idea}"

response = client.models.generate_content(
    model=MODEL,
    contents=search_prompt,
    config=types.GenerateContentConfig(
        tools=[search_tool],
    ),
)

print(response.text)


The market for AI applications in customer service, including tools that analyze customer support calls and automate follow-up emails, is experiencing substantial growth. The global AI customer service market was valued at approximately $12.06 billion in 2024 and is projected to reach between $47.82 billion by 2030 and $117.87 billion by 2034, with a compound annual growth rate (CAGR) of around 25%. The call center AI market specifically was estimated at $1.99 billion in 2024 and is expected to grow to $7.08 billion by 2030, with a CAGR of 23.8%. Other estimates place the call center AI market at $3.98 billion in 2025, reaching $30.69 billion by 2035. This growth is driven by the increasing demand for 24/7 customer support, the need to reduce operational costs, and the desire to enhance customer experience through personalized and efficient interactions.

Current competitors offering AI solutions that either directly or indirectly address the idea of analyzing customer support calls an

That should mention real, current companies instead of guessing from training data. Now let's
put everything together: rubric + JSON formatting + web search + all six categories in one go.


## Step 6: build the real thing

Same idea as before, just more categories and more detail in each one.


In [22]:
VALIDATION_SCHEMA = {
    "idea_summary": "one or two sentence neutral restatement of the idea",
    "overall_score": "integer 0-100",
    "verdict": "one of: Strong Go | Proceed with Caution | Pivot Needed | Not Recommended",
    "category_scores": {
        "problem_solution_fit": "integer 0-10",
        "market_opportunity": "integer 0-10",
        "competitive_landscape": "integer 0-10",
        "business_model": "integer 0-10",
        "technical_feasibility": "integer 0-10",
        "moat_differentiation": "integer 0-10",
    },
    "category_notes": {
        "problem_solution_fit": "one sentence justification",
        "market_opportunity": "one sentence justification",
        "competitive_landscape": "one sentence justification",
        "business_model": "one sentence justification",
        "technical_feasibility": "one sentence justification",
        "moat_differentiation": "one sentence justification",
    },
    "market_analysis": {
        "market_size": "brief estimate or description",
        "key_trends": "brief description",
        "target_customer": "brief description",
    },
    "competitors": [
        {"name": "string", "description": "one sentence", "threat_level": "Low | Medium | High"}
    ],
    "strengths": ["short phrase"],
    "weaknesses": ["short phrase"],
    "risks": ["short phrase"],
    "recommendations": ["short actionable phrase"],
}

# turning the schema dict into a JSON string so we can paste it inside the prompt text below
schema_text = json.dumps(VALIDATION_SCHEMA, indent=2)

SYSTEM_PROMPT = """You are a startup advisor and venture analyst who specializes in evaluating
generative AI products. Be direct and honest, not just encouraging - founders need real signal.

Research the idea using web search where it helps (current market size, real competitors, recent
news), then score it across six categories: problem-solution fit, market opportunity, competitive
landscape, business model, technical feasibility (given current gen AI limits like cost, latency,
and hallucination), and moat/differentiation (including the risk of foundation model providers
absorbing this feature themselves over time).

Respond with ONLY valid JSON matching this schema. No markdown fences, no commentary, just the JSON:

""" + schema_text

print("prompt is", len(SYSTEM_PROMPT), "characters")


prompt is 2045 characters


In [23]:
def validate_startup_idea(idea_text, use_web_search=True):
    """
    Sends the idea to Gemini and returns the parsed report as a python dict.
    Set use_web_search=False to skip live search (faster, but less current on
    market/competitor info).
    """
    config_kwargs = {"system_instruction": SYSTEM_PROMPT}

    if use_web_search:
        config_kwargs["tools"] = [types.Tool(google_search=types.GoogleSearch())]

    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=f"Startup idea:\n\n{idea_text}",
            config=types.GenerateContentConfig(**config_kwargs),
        )
    except Exception as e:
        if use_web_search:
            print("web search failed (", e, ") - trying again without it")
            return validate_startup_idea(idea_text, use_web_search=False)
        else:
            raise

    return extract_json(response.text)


print("validate_startup_idea() is ready")


validate_startup_idea() is ready


## Step 7: a quick manual test

Let's run one idea through it before building the nicer display/UI around it.


In [24]:
my_report = validate_startup_idea(my_idea)
print(json.dumps(my_report, indent=2))


{
  "idea_summary": "An AI application designed to listen to customer support calls, automatically generate a summary of the conversation, and then draft a follow-up email for the agent.",
  "overall_score": 68,
  "verdict": "Proceed with Caution",
  "category_scores": {
    "problem_solution_fit": 9,
    "market_opportunity": 7,
    "competitive_landscape": 4,
    "business_model": 7,
    "technical_feasibility": 8,
    "moat_differentiation": 5
  },
  "category_notes": {
    "problem_solution_fit": "The idea directly addresses a significant pain point for customer support agents: the substantial time spent on after-call work, including manual summarization and follow-up email drafting.",
    "market_opportunity": "The AI customer service market is large and growing rapidly, with a projected size of $15.12 billion in 2026 and significant expansion expected, indicating strong demand for efficiency-enhancing solutions.",
    "competitive_landscape": "The market is already crowded with e

## Step 8: make the output readable

Raw JSON is fine for code but ugly to read. Let's turn it into a nice markdown report.

First a tiny helper that turns a 0-10 score into a little text bar like `███████░░░`.


In [25]:
def score_bar(score, max_score=10, width=10):
    filled_blocks = round((score / max_score) * width)
    empty_blocks = width - filled_blocks
    return ("█" * filled_blocks) + ("░" * empty_blocks)


# quick check
print(score_bar(7))
print(score_bar(3))


███████░░░
███░░░░░░░


In [26]:
VERDICT_EMOJI = {
    "Strong Go": "🟢",
    "Proceed with Caution": "🟡",
    "Pivot Needed": "🟠",
    "Not Recommended": "🔴",
}


def render_report(idea_text, report):
    verdict = report.get("verdict", "Unknown")
    emoji = VERDICT_EMOJI.get(verdict, "⚪")

    lines = []
    lines.append(f"## {emoji} Verdict: {verdict} — Overall Score: {report.get('overall_score', '?')}/100\n")
    lines.append(f"**Idea:** {report.get('idea_summary', idea_text)}\n")

    lines.append("### Category Scores")
    for category, score in report.get("category_scores", {}).items():
        label = category.replace("_", " ").title()
        note = report.get("category_notes", {}).get(category, "")
        lines.append(f"- **{label}** `{score_bar(score)}` {score}/10 — {note}")

    market = report.get("market_analysis", {})
    lines.append("\n### Market Analysis")
    lines.append(f"- **Market size:** {market.get('market_size', 'n/a')}")
    lines.append(f"- **Key trends:** {market.get('key_trends', 'n/a')}")
    lines.append(f"- **Target customer:** {market.get('target_customer', 'n/a')}")

    lines.append("\n### Competitors")
    for comp in report.get("competitors", []):
        lines.append(f"- **{comp.get('name')}** ({comp.get('threat_level', '?')} threat) — {comp.get('description')}")

    lines.append("\n### Strengths")
    for item in report.get("strengths", []):
        lines.append(f"- {item}")

    lines.append("\n### Weaknesses")
    for item in report.get("weaknesses", []):
        lines.append(f"- {item}")

    lines.append("\n### Risks")
    for item in report.get("risks", []):
        lines.append(f"- {item}")

    lines.append("\n### Recommendations")
    for item in report.get("recommendations", []):
        lines.append(f"- {item}")

    full_markdown = "\n".join(lines)
    display(Markdown(full_markdown))


# try it on the report we already have
render_report(my_idea, my_report)


## 🟡 Verdict: Proceed with Caution — Overall Score: 68/100

**Idea:** An AI application designed to listen to customer support calls, automatically generate a summary of the conversation, and then draft a follow-up email for the agent.

### Category Scores
- **Problem Solution Fit** `█████████░` 9/10 — The idea directly addresses a significant pain point for customer support agents: the substantial time spent on after-call work, including manual summarization and follow-up email drafting.
- **Market Opportunity** `███████░░░` 7/10 — The AI customer service market is large and growing rapidly, with a projected size of $15.12 billion in 2026 and significant expansion expected, indicating strong demand for efficiency-enhancing solutions.
- **Competitive Landscape** `████░░░░░░` 4/10 — The market is already crowded with established CRM/CCaaS players (Salesforce, Zendesk) and specialized conversation intelligence platforms (Gong, Observe.AI) that offer similar or integrated features, making differentiation challenging.
- **Business Model** `███████░░░` 7/10 — A SaaS subscription model, likely tiered by user or usage, fits well with enterprise software procurement and offers predictable revenue streams, but pricing needs to be competitive against bundled offerings.
- **Technical Feasibility** `████████░░` 8/10 — Current generative AI technology is capable of accurate transcription, summarization, and drafting of human-like text, making the core functionality technically feasible, though ongoing refinement for accuracy is crucial.
- **Moat Differentiation** `█████░░░░░` 5/10 — 

### Market Analysis
- **Market size:** The global AI customer service market is projected at $15.12 billion in 2026, growing to $117.87 billion by 2034. The generative AI in customer services market is specifically projected at $603.94 million in 2025, reaching over $5.3 billion by 2035.
- **Key trends:** Rising demand for 24/7 support, digital transformation, the need to reduce operational costs, and increasing integration of generative AI into existing CRM platforms are key trends. There's a strong shift towards agent-assist tools and autonomous AI agents for end-to-end resolution.
- **Target customer:** Mid-to-large enterprises with significant customer support operations, contact centers, and sales teams seeking to improve agent productivity, reduce after-call work, and ensure consistent follow-up communication.

### Competitors
- **Salesforce Service Cloud (Einstein AI)** (High threat) — Integrates AI for case classification, reply recommendations, article generation, and work summaries directly within the Salesforce ecosystem.
- **Zendesk AI** (High threat) — Offers AI-powered tools like agent copilot, generative search, quick answers, AI-generated ticket summaries, and suggested replies built into their platform.
- **Gong.io / Observe.AI (Conversation Intelligence Platforms)** (High threat) — Leaders in recording, transcribing, and analyzing sales/service calls, providing insights, call summaries, and increasingly offering AI-generated follow-up drafts.
- **Fireflies.ai / Momentum / CallRail** (Medium threat) — Specialized AI tools that focus on meeting transcription, summarization, and automated drafting of follow-up emails, often with CRM integrations.

### Strengths
- Addresses a clear and quantifiable pain point (reducing after-call work).
- Leverages proven generative AI capabilities for text generation.
- Potential for significant productivity gains for customer support agents.
- Strong market growth in AI for customer service.

### Weaknesses
- High competition from established, integrated platforms.
- Risk of AI hallucinations requiring human review, impacting full automation.
- Dependency on accurate speech-to-text transcription.
- Integration challenges with diverse existing customer support systems.

### Risks
- Feature commoditization by larger players or foundation model APIs.
- Difficulty in achieving superior accuracy and reliability consistently.
- Customer skepticism towards AI-generated communication lacking human nuance.
- Data privacy and security concerns given sensitive call content.

### Recommendations
- Focus on deep integration with specific CRM/CCaaS platforms where competitors are weaker.
- Develop proprietary fine-tuned models for industry-specific terminology and communication styles to enhance accuracy and differentiation.
- Emphasize the human-in-the-loop workflow, highlighting AI as an assistant rather than a replacement to build trust and ensure quality.
- Explore advanced features beyond basic drafting, such as sentiment analysis-driven tone adjustment, proactive next-step suggestions, or integration with knowledge bases for richer email content.
- Target specific niches (e.g., highly technical support, regulated industries) where generic solutions may fall short.

## Step 9: keep track of every idea you test

So you can compare multiple ideas later, we'll save each result into a list as we go.


In [27]:
validation_history = []  # each entry: timestamp, short idea text, score, verdict, full report


def log_validation(idea_text, report):
    entry = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "idea": idea_text[:80] + ("..." if len(idea_text) > 80 else ""),
        "overall_score": report.get("overall_score"),
        "verdict": report.get("verdict"),
        "report": report,
    }
    validation_history.append(entry)


def show_history():
    if len(validation_history) == 0:
        print("nothing validated yet")
        return
    table_rows = []
    for entry in validation_history:
        table_rows.append({
            "timestamp": entry["timestamp"],
            "idea": entry["idea"],
            "overall_score": entry["overall_score"],
            "verdict": entry["verdict"],
        })
    df = pd.DataFrame(table_rows).sort_values("overall_score", ascending=False)
    display(df)


def export_report(index=-1, filename=None):
    """Saves one past report (default: the most recent) to a .json file."""
    if len(validation_history) == 0:
        print("nothing to export yet")
        return
    entry = validation_history[index]
    if filename is None:
        safe_time = entry["timestamp"].replace(":", "-")
        filename = f"validation_report_{safe_time}.json"
    with open(filename, "w") as f:
        json.dump(entry["report"], f, indent=2)
    print("saved to", filename)


# log the test idea from earlier so show_history() has something to show
log_validation(my_idea, my_report)
print("history tracking ready")


history tracking ready


## Step 10: a simple input box so you don't have to edit code every time

This makes a text box + button right in the notebook output.


In [28]:
idea_box = widgets.Textarea(
    placeholder="Describe your gen AI startup idea: the problem, who it's for, what you're building, and how it makes money...",
    layout=widgets.Layout(width="100%", height="140px"),
)

search_checkbox = widgets.Checkbox(value=True, description="Use live web search")
validate_button = widgets.Button(description="Validate Idea", button_style="primary")
output_box = widgets.Output()


def on_button_click(button):
    idea_text = idea_box.value.strip()
    with output_box:
        clear_output()
        if idea_text == "":
            print("type an idea first")
            return
        print("thinking... (can take 10-30 seconds, longer with web search on)")
        try:
            report = validate_startup_idea(idea_text, use_web_search=search_checkbox.value)
        except Exception as error:
            clear_output()
            print("something went wrong:", error)
            return
        clear_output()
        render_report(idea_text, report)
        log_validation(idea_text, report)


validate_button.on_click(on_button_click)

display(widgets.VBox([idea_box, search_checkbox, validate_button, output_box]))


## Step 11: compare ideas

Run the box above for as many ideas as you like, then run this to see them ranked.


In [ ]:
show_history()


## Where to go from here

- `export_report()` saves the most recent validation to a `.json` file
- swap `MODEL = "gemini-2.5-flash"` for `"gemini-2.5-pro"` if you want deeper (but paid) reasoning
- add a 7th category, like a financial sanity check (CAC/LTV, margin after inference costs)
- feed in a whole pitch deck PDF instead of typing a description
- ask Gemini to critique its own first answer as a second pass, for a sanity check
